# Media Index Database

#### Setup and Mock Data

In [6]:
import pathlib
import pydantic_settings

class Settings(pydantic_settings.BaseSettings):
    mongo_uri: str = "mongodb://127.0.0.1:27017/?directConnection=true"
    db_name: str = "test_files"
    test_zip_file_url: str

settings = Settings()
settings.mongo_uri, settings.db_name

('mongodb://127.0.0.1:27017/?directConnection=true', 'test_files')

In [7]:
from example_helpers import RemoteZipDownloader
zd = RemoteZipDownloader.from_url(settings.test_zip_file_url)
temp_path = zd.path

In [8]:
import mediatools
print(mediatools.scan_directory(temp_path).display())

/tmp/tmpj5gbaqrx/
├── battles/
│       ├── [V] how_to_battle.mp4
│       ├── [I] how_to_battle_thumb.jpg
│       ├── [F] how_to_battle.m4a
│       └── [F] how_to_battle.txt
└── totk_builds/
    ├── [V] op_builds.mp4
    ├── [I] op_builds_thumb.jpg
    ├── [F] op_builds.m4a
    └── [F] op_builds.txt


In [24]:
import pymongo

async with pymongo.AsyncMongoClient(settings.mongo_uri, serverSelectionTimeoutMS=2000) as client:
    media_index = await mediatools.MediaIndexDB.from_client(client[settings.db_name])

    mindex = await mediatools.MediaIndexDB.from_client(
        client = client[settings.db_name]
    )
    await mindex.update_directory_index(temp_path, verbose=True)
    await mindex.dirs.find_by_path(temp_path)

    print(await mindex.dirs.count(temp_path))
    print(await mindex.videos.count(temp_path))

    sub_dirs = await mindex.dirs.find_by_path_prefix(temp_path)
    for sdir in sub_dirs:
        print(f'{sdir.path}: \n\tsubdirs: {[sp for sp in sdir.subpaths]} \n\tvideos: {[vf for vf in sdir.video_files]} \n\timages: {[ifile for ifile in sdir.image_files]}')
        pass

Scanning directories:   0%|                                                   | 0/4 [00:00<?, ?it/s]

Indexing videos: 100%|███████████████████████████████████████████████| 2/2 [00:00<00:00, 649.93it/s]

4
2
/tmp/tmpj5gbaqrx: 
	subdirs: ['battles', 'totk_builds'] 
	videos: [] 
	images: []
/tmp/tmpj5gbaqrx/battles: 
	subdirs: ['totk_lynels'] 
	videos: [] 
	images: []
/tmp/tmpj5gbaqrx/battles/totk_lynels: 
	subdirs: [] 
	videos: ['how_to_battle.mp4'] 
	images: ['how_to_battle_thumb.jpg']
/tmp/tmpj5gbaqrx/totk_builds: 
	subdirs: [] 
	videos: ['op_builds.mp4'] 
	images: ['op_builds_thumb.jpg']
